# BTC ML Test

In [ ]:
from datetime import datetime

import research
import data

In [ ]:
research.set_seed(123)

sym = 'BTCUSDT'
interval = '1h'
max_lags = 4
forecast_horizon = 1
annualized_rate = research.sharpe_to_annualized_rate(interval, 365, 24)
start_date = datetime(2025, 1, 1, 0, 0)
end_date = datetime(2026, 1, 1, 0, 0)

data.download_date_range(sym, start_date, end_date)

In [ ]:
ts = research.load_ohlc_timeseries_range(sym, interval, start_date, end_date)
ts

In [ ]:
import polars as pl

research.load_timeseries_range(sym, interval, start_date, end_date, pl.col('price').quantile(0.5).alias('price_median'))

In [ ]:

research.plot_timeseries(ts, sym, 'close', interval)

In [ ]:
import altair

altair.data_transformers.enable('vegafusion')
research.plot_timeseries(ts, sym, 'close', interval, 'dynamic')

In [ ]:
ts = ts.with_columns((pl.col('close') / pl.col('close').shift(forecast_horizon)).log().alias('close_log_return'))
ts

In [ ]:
target = 'close_log_return'
lr = pl.col(target)
ts = ts.with_columns(
    lr.shift(forecast_horizon + 1).alias(f'{target}_t+{forecast_horizon + 1}'),
    lr.shift(forecast_horizon + 2).alias(f'{target}_t+{forecast_horizon + 2}'),
    lr.shift(forecast_horizon + 3).alias(f'{target}_t+{forecast_horizon + 3}'),
    lr.shift(forecast_horizon + 4).alias(f'{target}_t+{forecast_horizon + 4}')
)
ts

In [ ]:
ts = research.add_lags(ts, target, max_lags, forecast_horizon)
ts

In [ ]:
ts = ts.drop_nulls()
ts

In [ ]:
research.plot_distribution(ts, target, no_bins = 100)

In [ ]:
research.plot_distribution(ts, 'close', no_bins = 100)
# This is why we use log returns instead of raw prices.
# The distribution of log returns is more Gaussian-like, which is a common assumption in many financial models.

In [ ]:
import torch.nn as nn

class LinearModel(nn.Module):
    def __init__(self, input_features):
        super(LinearModel, self).__init__()
        self.linear = nn.Linear(input_features, 1)

    def forward(self, x):
        # this function defines the forward pass of the model, 
        # which is how the input data flows through the model to produce an output
        return self.linear(x)
    

In [ ]:
input_features = 1
linear_model = LinearModel(input_features)

# Count all parameters in the model
total_params = sum(p.numel() for p in linear_model.parameters())

# Count only parameters that will be updated during training
trainable_params = sum(
    p.numel() for p in linear_model.parameters() if p.requires_grad
)

# Print formatted model information
print(f"\n{'='*60}")
print(f"{LinearModel.__name__}")
print(f"{'='*60}")
print(f"\nArchitecture:")
print(f"  {linear_model}")
print(f"\nParameter Count:")
print(f"  Total parameters:      {total_params:,}")
print(f"  Trainable parameters:  {trainable_params:,}")

# Warn if some parameters are frozen
if total_params != trainable_params:
    frozen_params = total_params - trainable_params
    print(f"  Frozen parameters:     {frozen_params:,}")
    print(f"\n  ⚠️  Note: {frozen_params:,} parameters are frozen")

print(f"{'='*60}\n")

for name, param in linear_model.named_parameters():
    if param.requires_grad:
        print(f"{name}:\n{param.data.numpy()}")

# Should be y = mx + b, where m is the slope and b is the y-intercept.


In [ ]:
features = ['close_log_return_lag_1']
target = 'close_log_return'
test_size = 0.25

print(f"Sample: {len(ts)}")
print(f"Test Size: {int(len(ts) * test_size)}")

split_idx = int(len(ts) * (1 - test_size))
print(f"Split Index: {split_idx}")

In [ ]:
ts_train, ts_test = ts[:split_idx], ts[split_idx:]
print(f"Training Sample: {len(ts_train)}")
print(f"Testing Sample: {len(ts_test)}")

ts_train

In [ ]:
ts_test

In [ ]:
import torch

x_train = torch.tensor(ts_train[features].to_numpy(), dtype=torch.float32)
x_test = torch.tensor(ts_test[features].to_numpy(), dtype=torch.float32)
y_train = torch.tensor(ts_train[target].to_numpy(), dtype=torch.float32)
y_test = torch.tensor(ts_test[target].to_numpy(), dtype=torch.float32)


In [ ]:
x_train

In [ ]:
x_test.shape

In [ ]:
y_train

In [ ]:
y_train.shape

In [ ]:
y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

y_train.shape

In [ ]:
# make the above re-usable
x_train, x_test, y_train, y_test = research.timeseries_train_test_split(ts, features, target, test_size)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

In [ ]:
no_epochs = 5000 # we can adjust this number based on how well the model is learning. More epochs can lead to better learning, but also risk overfitting.
lr = 0.0005 # learning rate, which controls how much to change the model in response to the estimated error each time the model weights are updated.

model = LinearModel(input_features)
criterion = nn.MSELoss()  # Mean Squared Error loss
optimiser = torch.optim.Adam(model.parameters(), lr=lr)

print(f"Training model for {no_epochs} epochs with learning rate {lr}...")

for epoch in range(no_epochs):
    y_hat = model(x_train)  # Forward pass: Compute predicted y by passing x to the model
    loss = criterion(y_hat, y_train)  # Compute and print loss

    # Zero gradients, perform a backward pass, and update the weights
    optimiser.zero_grad() # clear the old gradients
    loss.backward() # compute the gradients
    optimiser.step() # update the weights

    train_loss = loss.item() # get the scalar value of the loss for logging
    if epoch % 100 == 0:
        print(f"Epoch: [{epoch}/{no_epochs}], Loss: {train_loss}")

print("Training complete.")

for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}:\n{param.data.numpy()}")

model.eval()  # Set the model to evaluation mode
with torch.no_grad():  # Disable gradient calculation for inference
    y_hat = model(x_test)  # Get predictions for the test set
    test_loss = criterion(y_hat, y_test).item()  # Calculate the test loss
    print(f"Test Loss: {test_loss} | Train Loss: {train_loss}") # pyright: ignore[reportPossiblyUnboundVariable]

In [ ]:
trade_results = pl.DataFrame({
    'y_hat': y_hat.numpy().flatten(),
    'y': y_test.numpy().flatten()
}).with_columns(
    (pl.col('y_hat').sign() == pl.col('y').sign()).alias('is_won'),
    pl.col('y_hat').sign().alias('signal'),
).with_columns(
    (pl.col('signal') * pl.col('y')).alias('trade_log_return')
).with_columns(
    pl.col('trade_log_return').cum_sum().alias('equity_curve')
)
trade_results

In [ ]:
research.plot_line(trade_results, 'equity_curve')

In [ ]:
trade_results = trade_results.with_columns(
    (pl.col('equity_curve') - pl.col('equity_curve').cum_max()).alias('log_drawdown')
)
trade_results

In [ ]:
import numpy as np

max_drawdown_log = trade_results['log_drawdown'].min()
assert isinstance(max_drawdown_log, (int, float))
drawdown_pct = np.exp(max_drawdown_log) - 1
print(f"Max Drawdown (log): {max_drawdown_log:.4f}")
print(f"Max Drawdown (%): {drawdown_pct:.2%}")

win_rate = trade_results['is_won'].mean()
avg_win = trade_results.filter(pl.col('is_won') == True)['trade_log_return'].mean()
avg_loss = trade_results.filter(pl.col('is_won') == False)['trade_log_return'].mean()

# fix typing issues
assert isinstance(win_rate, (int, float))
assert isinstance(avg_win, (int, float))
assert isinstance(avg_loss, (int, float))
print(f"Win Rate: {win_rate:.2%}")

print(f"Average Win: {avg_win:.4f}")
print(f"Average Loss: {avg_loss:.4f}")

ev = (win_rate * avg_win) + ((1 - win_rate) * avg_loss)
print(f"Expected Value (EV): {ev:.4f}")

In [ ]:
total_log_return = trade_results['trade_log_return'].sum()
total_log_return

In [ ]:
# take it out of log space
assert isinstance(total_log_return, (int, float))
compound_return = np.exp(total_log_return) - 1
print(f"Compound Return: {compound_return:.4f}")
print(f"Profit/Loss on $1000: ${1000 * compound_return:.2f}")

In [ ]:
equity_trough = trade_results['equity_curve'].min()
equity_peak = trade_results['equity_curve'].max()
print(f"Equity Peak: {equity_peak:.4f}")
print(f"Equity Trough: {equity_trough:.4f}")

In [ ]:
std = trade_results['trade_log_return'].std()
print(f"Standard Deviation of Returns: {std:.4f}"
      )
assert isinstance(std, (int, float))
sharpe_ratio = ev / std * annualized_rate
print(f"Sharpe Ratio: {sharpe_ratio:.4f}")

In [ ]:
perf = research.eval_model_performance(y_test, y_hat, features, target, annualized_rate, log=True)

In [ ]:
target = 'close_log_return'
features = ['close_log_return_lag_2']
model = LinearModel(input_features)
perf = research.benchmark_model_performance(ts, features, target, model, annualized_rate, no_epochs=50, log=True)

In [ ]:
import itertools

benchmarks = []
feature_pool = [f'{target}_lag_{i}' for i in range(1, max_lags + 1)]
combos = list(itertools.combinations(feature_pool, 1))

for features in combos:
    model = LinearModel(input_features)
    perf = research.benchmark_model_performance(ts, list(features), target, model, annualized_rate, no_epochs=200, loss=nn.L1Loss())
    benchmarks.append(perf)

benchmark = pl.DataFrame(benchmarks).sort('sharpe', descending=True)
benchmark

In [ ]:
research.auto_reg_corr_matrx(ts, target, max_lags)

In [ ]:
features = ['close_log_return_lag_3']
model = LinearModel(input_features)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
model_trades = research.learn_model_trades(ts, features, target, model, no_epochs=200, loss=nn.L1Loss(), optimizer=optimizer)
research.plot_line(model_trades, 'equity_curve')

# This model has positive returns.. but probably not with fees

In [ ]:
# e.g. VIP 4 level on binance
maker_fee = 0.0001
taker_fee = 0.0003

In [ ]:
roundtrip_fee_log = np.log(1 - 2 * taker_fee)

model_trades = model_trades.with_columns(pl.lit(roundtrip_fee_log).alias('tx_fee_log'))
model_trades = model_trades.with_columns((pl.col('trade_log_return') + pl.col('tx_fee_log')).alias('trade_log_return_net'))
model_trades = model_trades.with_columns(pl.col('trade_log_return_net').cum_sum().alias('equity_curve_net'))

model_trades # now making a net loss

In [ ]:
research.plot_line(model_trades, 'equity_curve_net')

In [ ]:
model_trades['is_won'].mean() # still wins > 50%

In [ ]:
model_trades = research.add_tx_fees_log(model_trades, maker_fee, taker_fee)
model_trades

In [ ]:
time_interval = '6h'
annualized_rate = research.sharpe_to_annualized_rate(time_interval, 365, 24)
ts = research.load_ohlc_timeseries_range(sym, time_interval, start_date, end_date)
ts

In [ ]:
no_lags = 3
ts = research.add_log_return_features(ts, 'close', forecast_horizon, max_no_lags=no_lags)
ts

In [ ]:
target = 'close_log_return'
feature_pool = [f'{target}_lag_{i}' for i in range(1, no_lags + 1)]
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, loss=nn.HuberLoss())

In [ ]:
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, loss=nn.MSELoss())

In [ ]:
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, loss=nn.L1Loss())

In [ ]:
research.auto_reg_corr_matrx(ts.drop_nulls(), target, no_lags)

In [ ]:
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, loss=nn.MSELoss())

In [ ]:
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, loss=nn.L1Loss(), test_size=0.3)

In [ ]:
features = ['close_log_return_lag_1']
model = LinearModel(len(features))
model_trades = research.learn_model_trades(ts.drop_nulls(), features, target, model, loss=nn.L1Loss())
model_trades = research.add_tx_fees_log(model_trades, maker_fee, taker_fee)
research.plot_line(model_trades, 'equity_curve')

In [ ]:
time_interval = '12h'
annualized_rate = research.sharpe_to_annualized_rate(time_interval, 365, 24)

no_lags = 4
ts = research.load_ohlc_timeseries_range(sym, time_interval, start_date, end_date)
ts = research.add_log_return_features(ts, 'close', forecast_horizon, max_no_lags=no_lags)
ts

In [ ]:
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, max_no_features=3, loss=nn.MSELoss())

In [ ]:
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, max_no_features=3, loss=nn.HuberLoss())

In [ ]:
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, max_no_features=3, loss=nn.L1Loss(), test_size=0.25)

In [ ]:
features = ['close_log_return_lag_1','close_log_return_lag_2','close_log_return_lag_3']
model = LinearModel(len(features))
model_trades = research.learn_model_trades(ts.drop_nulls(), features, target, model, loss=nn.L1Loss())
model_trades = research.add_tx_fees_log(model_trades, maker_fee, taker_fee)
research.plot_line(model_trades, 'equity_curve')